In [6]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F
from encoder import SmallCNNEncoder


In [2]:
from torch.utils.data import Dataset

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        x1 = self.base_transform(x)
        x2 = self.base_transform(x)
        return x1, x2

class MRISSLWrapper(Dataset):
    def __init__(self, base_dataset, two_crop_transform):
        self.base = base_dataset
        self.t = two_crop_transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        out = self.base[idx]   # puede ser (x,y) o (x,y,...) o dict
        if isinstance(out, dict):
            x = out["x"] if "x" in out else out["image"]
        else:
            x = out[0]         # siempre el primer elemento es x (en casi todos los datasets)

        x1, x2 = self.t(x)
        return x1, x2


In [4]:

class SimCLR(nn.Module):
    def __init__(self, in_ch=9, feat_dim=256, proj_dim=256):
        super().__init__()
        self.encoder = SmallCNNEncoder(in_ch=in_ch, feat_dim=feat_dim)
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(),
            nn.Linear(feat_dim, proj_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return z


In [5]:
def nt_xent_loss(z1, z2, temperature=0.2):
    """
    z1, z2: (B, D) ya normalizados
    """
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # (2B, D)

    # similitudes
    sim = torch.mm(z, z.t()) / temperature  # (2B, 2B)

    # máscara para quitar diagonal
    mask = torch.eye(2*B, device=z.device).bool()
    sim.masked_fill_(mask, -1e9)

    # positivos: i <-> i+B
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)  # (2B,)

    # denominador: logsumexp de cada fila
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()


In [8]:
import torch
from torch.utils.data import DataLoader
from dataloader import train_set, train_tf, df_train
from mri_dataset import MRIPatient25DDataset
device = "cuda" if torch.cuda.is_available() else "cpu"

# base_dataset = MRIPatient25DDataset(df_train, transform=..., cache=True)
# OJO: para SimCLR, el transform debe incluir augmentations "fuertes".
two_crop = TwoCropsTransform(base_transform=train_tf)
base_ssl = MRIPatient25DDataset(df_train, transform=None, cache=True)
ssl_dataset = MRISSLWrapper(base_ssl, TwoCropsTransform(train_tf))
ssl_loader = DataLoader(ssl_dataset, batch_size=64, shuffle=True, num_workers=0)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2072 entries, 0 to 2071
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pid             2072 non-null   object 
 1   pCR             1432 non-null   float64
 2   n_xy            2072 non-null   float64
 3   n_z             2072 non-null   float64
 4   n_times         2072 non-null   float64
 5   pre             2072 non-null   float64
 6   post_early      2072 non-null   float64
 7   post_late       2072 non-null   float64
 8   slice_thick     2072 non-null   float64
 9   xy_spacing      2072 non-null   float64
 10  mask_start      2072 non-null   float64
 11  mask_end        2072 non-null   float64
 12  sraw            2072 non-null   float64
 13  eraw            2072 non-null   float64
 14  scol            2072 non-null   float64
 15  ecol            2072 non-null   float64
 16  tum_vol         2071 non-null   float64
 17  age             2072 non-null   f

NameError: name 'TwoCropsTransform' is not defined

In [7]:

model = SimCLR(in_ch=9).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("PyTorch version:", torch.__version__)
print("Number of GPUs:", torch.cuda.device_count())


CUDA available: True
CUDA version: 12.1
PyTorch version: 2.5.1+cu121
Number of GPUs: 1


In [9]:

model.train()
for epoch in range(50):
    total = 0.0
    for x1, x2 in ssl_loader:
        x1, x2 = x1.to(device), x2.to(device)
        z1 = model(x1)
        z2 = model(x2)
        loss = nt_xent_loss(z1, z2, temperature=0.5)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"epoch {epoch:03d} loss {total/len(ssl_loader):.4f}")


['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ISPY2-460812
['ISPY2-460812_spy2_vis1_dce_aqc_0.nii.gz', 'ISPY2-460812_spy2_vis1_dce_aqc_2.nii.gz', 'ISPY2-460812_spy2_vis1_dce_aqc_5.nii.gz']
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-928446
['ACRIN-6698-928446_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-928446_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-928446_spy2_vis1_dce_aqc_6.nii.gz']
['ACRIN-6698-102212_

In [10]:
sample = train_set[0]
print(type(sample))
if isinstance(sample, tuple):
    print("tuple len:", len(sample))
    for i, s in enumerate(sample):
        if hasattr(s, "shape"):
            print(i, "shape:", s.shape)
        else:
            print(i, type(s))
elif isinstance(sample, dict):
    print(sample.keys())


['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-103939
['ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz']
<class 'tuple'>
tuple len: 3
0 shape: torch.Size([9, 256, 256])
1 shape: torch.Size([1])
2 <class 'str'>


In [ ]:
torch.save(model.encoder.state_dict(), "results/ssl.pth")


In [8]:
# Modelo supervisado
class MRIClassifier(nn.Module):
    def __init__(self, encoder, feat_dim=256):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(feat_dim, 1)  # binario

    def forward(self, x):
        h = self.encoder(x)
        logits = self.classifier(h)
        return logits.squeeze(1)


In [9]:
# cargar pesos SSL
encoder = SmallCNNEncoder(in_ch=9, feat_dim=256)
encoder.load_state_dict(torch.load("results/ssl.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MRIClassifier(encoder).to(device)


C:\Users\Ivón\AppData\Local\Temp\ipykernel_27044\3553371412.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("results/ssl.pth"))


In [14]:
#Finetuning
for p in model.encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-4},
])


In [15]:
# Loss
lossBCE = nn.BCEWithLogitsLoss()
#Training
def train_epoch(model, loader, optimizer):
    model.train()
    for x, y in loader:
        x = x.to(device)
        y = y.float().to(device)
        logits = model(x)
        loss = lossBCE(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [11]:
# Evaluation
from sklearn.metrics import roc_auc_score

def eval_auc(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(device)
            y = y.float().to(device)
            probs = torch.sigmoid(model(x))
            ys.append(y.cpu())
            ps.append(probs.cpu())
    return roc_auc_score(torch.cat(ys), torch.cat(ps))


In [12]:
# cargar pesos SSL
encoder = SmallCNNEncoder(in_ch=9, feat_dim=256)
encoder.load_state_dict(torch.load("results/ssl.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MRIClassifier(encoder).to(device)


C:\Users\Ivón\AppData\Local\Temp\ipykernel_27044\3553371412.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("results/ssl.pth"))


In [13]:
# Validation

from dataloader import val_loader
auc_val = eval_auc(model, val_loader)
print("Val AUC:", auc_val)


['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-138027
['ACRIN-6698-138027_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-138027_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-138027_spy2_vis1_dce_aqc_5.nii.gz']
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-156163
['ACRIN-6698-156163_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-156163_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-156163_spy2_vis1_dce_aqc_6.nii.gz']


In [14]:
# Test
from dataloader import test_loader

auc_test = eval_auc(model, test_loader)
print("Test AUC:", auc_test)


['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-102212
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz']
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-116603
['ACRIN-6698-116603_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-116603_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-116603_spy2_vis1_dce_aqc_6.nii.gz']


In [16]:
import numpy as np
model.eval()

p_img_val = []
y_val_list = []

with torch.no_grad():
    for xb, yb, _ in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        p_img_val.extend(probs)
        y_val_list.extend(yb.numpy())

p_img_val = np.array(p_img_val)
y_val = np.array(y_val_list)


In [18]:
p_img_test = []
y_test_list = []

model.eval()
with torch.no_grad():
    for xb, yb, _ in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        p_img_test.extend(probs)
        y_test_list.extend(yb.numpy())

p_img_test = np.array(p_img_test)
y_test = np.array(y_test_list)


In [9]:
# cargar pesos SSL
encoder = SmallCNNEncoder(in_ch=9, feat_dim=256)
encoder.load_state_dict(torch.load("results/ssl.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MRIClassifier(encoder).to(device)


C:\Users\Ivón\AppData\Local\Temp\ipykernel_16592\3553371412.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("results/ssl.pth"))


In [12]:
# ----- PREDICT ON TEST -----
from dataloader import test_loader
import numpy as np
from sklearn.metrics import roc_auc_score
all_probs = []
all_y = []
all_pid = []

with torch.no_grad():
    for xb, yb, pid in test_loader:   
        xb = xb.to(device)
        yb = yb.float().to(device)

        logits = model(xb)                    # (B,1) logits
        probs  = torch.sigmoid(logits)        # (B,1) prob pCR
        preds  = (probs >= 0.5).long()        # (B,1) class 0/1

        all_probs.append(probs.view(-1).cpu().numpy())
        all_y.append(yb.view(-1).cpu().numpy())
        all_pid.extend(list(pid))

        # imprimir algunas predicciones (opcional)
        for p_hat, y_true in zip(preds.view(-1), yb.view(-1).long()):
            print("Predicted class:", int(p_hat.item()), "Real class:", int(y_true.item()))

all_probs = np.concatenate(all_probs)
all_y = np.concatenate(all_y)

print("Test probs min/mean/max:", all_probs.min(), all_probs.mean(), all_probs.max())
print("Test AUC:", roc_auc_score(all_y, all_probs))


Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 1 Real class: 1
Predicted class: 0 Real class: 1
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 1
Predicted class: 1 Real class: 0
Predicted class: 0 Real class: 1
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 1 Real class: 1
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 1 Real class: 1
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 1 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 1
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted class: 0 Real class: 0
Predicted 

In [14]:
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
# --- PATIENT-LEVEL AGGREGATION ---
df = pd.DataFrame({
    'patient_id': all_pid,
    'prob':       all_probs,
    'label':      all_y
})

# Agregar por paciente (media de probs, label es único por paciente)
patient_df = df.groupby('patient_id').agg(
    prob  = ('prob',  'mean'),
    label = ('label', 'first')  # el label es el mismo para todos los slices del paciente
).reset_index()

# --- METRICS ---
auc_patient = roc_auc_score(patient_df['label'], patient_df['prob'])
print("CNN (no SSL) AUC (patient-level):", auc_patient)
print(classification_report(patient_df['label'], patient_df['prob'] > 0.5))

CNN (no SSL) AUC (patient-level): 0.582089552238806
              precision    recall  f1-score   support

         0.0       0.70      0.87      0.77        67
         1.0       0.44      0.22      0.29        32

    accuracy                           0.66        99
   macro avg       0.57      0.54      0.53        99
weighted avg       0.61      0.66      0.62        99

